In [2]:
import numpy as np
import pandas as pd

## Round-Robin Miner Assignment

In [3]:
csv_path = '../data/bitcoin_blocks.csv'

df = pd.read_csv(csv_path)
df.head()

,block_id,n_transactions,size,difficulty,transaction_volume
0,0,619,1292812,1.000000e+00,5.139540e+03
1,1,5368,974804,1.233186e+07,1.235600e+06
2,2,4540,1916683,2.466371e+07,1.699461e+06
3,3,3044,1433631,3.699557e+07,1.783661e+06
4,4,3003,479187,4.932742e+07,1.601429e+06


In [4]:
# def preprocess(csv_path, n_miners=100):
#     df = pd.read_csv(csv_path)
#     df['miner_id'] = df['block_id'] % n_miners
#     return df

In [5]:
# Round-robin miner assignment for each block
df['miner_id'] = df['block_id'] % 100
df.head(10)

,block_id,n_transactions,size,difficulty,transaction_volume,miner_id
0,0,619,1292812,1.000000e+00,5.139540e+03,0
1,1,5368,974804,1.233186e+07,1.235600e+06,1
2,2,4540,1916683,2.466371e+07,1.699461e+06,2
3,3,3044,1433631,3.699557e+07,1.783661e+06,3
4,4,3003,479187,4.932742e+07,1.601429e+06,4
5,5,5955,294607,6.165928e+07,1.250893e+05,5
6,6,597,204560,7.399113e+07,6.052177e+05,6
7,7,4837,1717723,8.632299e+07,2.574254e+05,7
8,8,1398,470837,9.865484e+07,8.240118e+05,8
9,9,654,1513868,1.109867e+08,1.184718e+06,9


In [6]:
# loops back to miner_id 0 at block_id 100
df.iloc[97:105][['block_id', 'miner_id']]

,block_id,miner_id
97,97,97
98,98,98
99,99,99
100,100,0
101,101,1
102,102,2
103,103,3
104,104,4


## Aggregation

In [7]:
# usually transaction fees are a percentage of each transaction
# for simplicity, we just use transactions * transaction volume without multiplying a percentage since the ratios would be the same
df['fee_proxy'] = df['n_transactions'] * df['transaction_volume']
df

,block_id,n_transactions,size,difficulty,transaction_volume,miner_id,fee_proxy
0,0,619,1292812,1.000000e+00,5.139540e+03,0,3.181375e+06
1,1,5368,974804,1.233186e+07,1.235600e+06,1,6.632700e+09
2,2,4540,1916683,2.466371e+07,1.699461e+06,2,7.715552e+09
3,3,3044,1433631,3.699557e+07,1.783661e+06,3,5.429464e+09
4,4,3003,479187,4.932742e+07,1.601429e+06,4,4.809091e+09
...,...,...,...,...,...,...,...
810904,810904,2393,1532604,9.999951e+12,7.826699e+05,4,1.872929e+09
810905,810905,4158,649434,9.999963e+12,7.708996e+05,5,3.205401e+09
810906,810906,6181,11369,9.999975e+12,6.373446e+05,6,3.939427e+09
810907,810907,907,751999,9.999988e+12,6.624588e+05,7,6.008501e+08


In [8]:
minerdf_test = df.groupby('miner_id')
# groupby doesnt create a dataframe yet, it just tells pandas we're grouping by this
# 'miner_id' but we havent told it what to put in the columns and how to calculate
# the new dataframe data from the original dataframe.
type(minerdf_test)

pandas.api.typing.DataFrameGroupBy

In [9]:
miner_df = df.groupby('miner_id').agg(
    blocks_mined=('block_id', 'count'),
    avg_transactions=('n_transactions', 'mean'),
    avg_volume=('transaction_volume', 'mean'),
    avg_fee=('fee_proxy', 'mean'),
    fee_volatility=('fee_proxy', 'std'),
    avg_block_size=('size', 'mean'),
    difficulty=('difficulty', 'mean'),
    profitability=('fee_proxy', 'sum'),
).reset_index()

# convert profitability from total sum to avg profitability
miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
miner_df

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,profitability
0,0,8110,3432.249568,1.049907e+06,3.613344e+09,3.238619e+09,1.000129e+06,4.999951e+12,3.612899e+09
1,1,8110,3437.731936,1.043139e+06,3.591012e+09,3.167694e+09,1.005087e+06,4.999963e+12,3.590569e+09
2,2,8110,3437.041060,1.056719e+06,3.633002e+09,3.187869e+09,1.012849e+06,4.999975e+12,3.632554e+09
3,3,8110,3486.842663,1.046138e+06,3.633910e+09,3.186243e+09,1.003623e+06,4.999988e+12,3.633462e+09
4,4,8110,3433.820222,1.052398e+06,3.623335e+09,3.213886e+09,9.909934e+05,5.000000e+12,3.622888e+09
...,...,...,...,...,...,...,...,...,...
95,95,8109,3490.174867,1.050761e+06,3.670072e+09,3.228316e+09,9.952050e+05,5.000506e+12,3.669619e+09
96,96,8109,3469.832408,1.055151e+06,3.636558e+09,3.198101e+09,1.002803e+06,5.000518e+12,3.636110e+09
97,97,8109,3467.445184,1.042264e+06,3.634216e+09,3.224168e+09,9.969971e+05,5.000530e+12,3.633768e+09
98,98,8109,3469.489826,1.061009e+06,3.692416e+09,3.229356e+09,9.897764e+05,5.000543e+12,3.691961e+09


## Calculating Efficiency Score and Binary Label

In [ ]:
# added very small number (1 * 10^-9) to prevent division by zero
# if any miner has avg_volume of 0
miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
miner_df.sort_values(by='efficiency', ascending=False) # show by descending efficiency

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,profitability,efficiency
28,28,8109,3482.256752,1.033372e+06,3.596305e+09,3.170148e+09,1.002843e+06,4.999679e+12,3.595862e+09,0.003370
29,29,8109,3520.568011,1.047628e+06,3.688860e+09,3.219243e+09,1.000667e+06,4.999692e+12,3.688405e+09,0.003361
67,67,8109,3472.035516,1.033257e+06,3.604440e+09,3.182318e+09,1.009763e+06,5.000160e+12,3.603996e+09,0.003360
6,6,8110,3497.657707,1.041049e+06,3.651006e+09,3.171610e+09,1.007461e+06,5.000025e+12,3.650556e+09,0.003360
77,77,8109,3480.787273,1.036851e+06,3.603228e+09,3.193937e+09,1.001647e+06,5.000284e+12,3.602784e+09,0.003357
...,...,...,...,...,...,...,...,...,...,...
2,2,8110,3437.041060,1.056719e+06,3.633002e+09,3.187869e+09,1.012849e+06,4.999975e+12,3.632554e+09,0.003253
45,45,8109,3427.352695,1.054067e+06,3.598595e+09,3.192410e+09,9.989339e+05,4.999889e+12,3.598152e+09,0.003252
18,18,8109,3430.300530,1.056533e+06,3.634489e+09,3.216703e+09,1.001063e+06,4.999556e+12,3.634041e+09,0.003247
68,68,8109,3438.549760,1.059645e+06,3.623344e+09,3.203651e+09,9.965575e+05,5.000173e+12,3.622897e+09,0.003245


In [19]:
# Calculate median efficiency
median_efficiency = miner_df['efficiency'].median()
print(f"Median Efficiency = {median_efficiency}")
# label miner as 1 if its efficiency is more than median, else 0
miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
miner_df[['efficiency', 'label']].head(10)

Median Efficiency = 0.0033018258364578886


,efficiency,label
0,0.003269,0
1,0.003296,0
2,0.003253,0
3,0.003333,1
4,0.003263,0
5,0.003301,0
6,0.003360,1
7,0.003352,1
8,0.003346,1
9,0.003304,1


In [20]:
miner_df['label'].value_counts()

label
0    50
1    50
Name: count, dtype: int64

## Train/Test Split

In [26]:
feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                   'avg_fee', 'fee_volatility', 'avg_block_size', 
                   'difficulty', 'profitability', 'efficiency']

X = miner_df[feature_columns]
y = miner_df['label']

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [31]:
print(X_train.shape)
print(X_test.shape)

(80, 9)
(20, 9)
